# Tips 21：自由記述を分類して要約

**業務の困りごと**：アンケートの自由記述を読み解き、傾向を報告する。

**After**：Colabでサンプルを作り、処理して結果を確認します。

**使用ライブラリ**：pandas（生成AIへの発展は任意）

### ファイルの場所
`/content/` はColabの一時領域です。PCやGoogle Driveには自動保存されません。左側のフォルダから結果を選んでダウンロードできます。セッション終了前に保存してください。

上から順番に実行してください。サンプルは各ノートブック内で作ります。再実行時は同名の練習ファイルを上書きします。


## 1. 練習用データと準備
サンプルを生成します。実務では、この入力を自分のファイルに置き換えます。


In [ ]:
import pandas as pd
from pathlib import Path
base=Path('/content') if Path('/content').exists() else Path.cwd()
responses=pd.DataFrame({'コメント':['予約が簡単で便利','待ち時間が長い','スタッフの説明が丁寧','予約しづらい','待ち時間が短くて助かる','説明が分かりにくい']})


## 2. 自動処理
入力を読み込み、目的の形式で保存します。


In [ ]:
# まずは決めた語で分類する。複数に該当する場合は先に見つかったテーマを採用。
rules={'予約':['予約'],'待ち時間':['待ち'],'接客・説明':['説明','スタッフ']}
def classify(text):
    return next((label for label,words in rules.items() if any(w in text for w in words)),'その他')
responses['分類']=responses['コメント'].map(classify)
summary=responses.groupby('分類',as_index=False).agg(件数=('コメント','size'),例=('コメント','first'))
out=base/'Tips21_自由記述分類.xlsx'
with pd.ExcelWriter(out) as writer:
    responses.to_excel(writer,'分類結果',index=False)
    summary.to_excel(writer,'集計',index=False)
print(summary)


## 3. 件数・内容の確認
入力と出力が意図した内容か確認します。


In [ ]:
assert summary['件数'].sum()==len(responses)
print('確認OK:',out)


## 発展：生成AIで分類・要約する
Pythonで個人情報を除き、コメントを少量ずつ確認してから、許可されたAIサービスに渡します。例えば次の指示文を使います。

> 次の自由記述を「予約」「待ち時間」「接客・説明」「その他」に分類し、各分類の主な肯定意見と改善要望を1文ずつ要約してください。判定が難しいものは「要確認」とし、元のコメント番号を残してください。

AIの結果は人が抜き取り確認し、分類件数が元データと一致するか調べます。この教材は外部APIへの送信を実行しません。


## 4. 持ち帰り・練習

**練習問題**：「料金」の分類ルールを追加し、該当するコメントを作ってみましょう。

これはキーワードに基づく分類です。生成AIによる意味分類や要約は別工程で、内容の確認と個人情報の扱いが必要です。

出力をPCへ保存したい場合は次のセルをColabで実行します。Colab以外では出力場所を表示します。


In [ ]:
try:
    from google.colab import files
    files.download(str(out))
except ImportError:
    print('出力ファイル:',out.resolve())
